## Libraries

In [1]:
import random
import numpy as np
import torch
from pathlib import Path
import kagglehub

/home/harsh/Work/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data 

In [2]:
DATASET = "jessicali9530/celeba-dataset"
EXPECTED_IMAGES = 202_599

# Relative to the notebook (Projects/Notebooks), so it works on any machine
DATA_DIR = (Path.cwd().parent / "Data" / "celeba").resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)

def count_jpgs(root):
    return sum(1 for _ in root.rglob("*.jpg"))

existing = count_jpgs(DATA_DIR)

if existing >= EXPECTED_IMAGES:
    print(f"Dataset already present ({existing:,} images). Skipping download.")
else:
    print(f"Found {existing:,} images, expected {EXPECTED_IMAGES:,}. Downloading...")
    kagglehub.dataset_download(
        DATASET,
        output_dir=str(DATA_DIR),
        force_download=existing > 0,   # re-fetch only if a partial copy exists
    )

# Locate the folder that actually holds the JPEGs (the zip may nest it twice)
first = next(DATA_DIR.rglob("*.jpg"), None)
if first is None:
    raise FileNotFoundError(f"No JPEGs found under {DATA_DIR}. Download failed.")

IMG_DIR = first.parent
n_images = sum(1 for _ in IMG_DIR.glob("*.jpg"))
if n_images < EXPECTED_IMAGES:
    raise RuntimeError(f"Only {n_images:,} images in {IMG_DIR}. Delete {DATA_DIR} and rerun.")

print(f"Ready: {n_images:,} images in {IMG_DIR}")


Dataset already present (202,599 images). Skipping download.
Ready: 202,599 images in /home/harsh/Work/Notebooks/Projects/Data/celeba/img_align_celeba/img_align_celeba


In [3]:
# Checks For GPU
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)   # safe to call even without a GPU

# Falls back to CPU so other people can still run the notebook
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Device:", device)
if device.type == "cuda":
    free, total = torch.cuda.mem_get_info()
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA (built with):", torch.version.cuda)
    print(f"VRAM free/total: {free / 1e9:.1f} / {total / 1e9:.1f} GB")

PyTorch: 2.13.0+cu130
Device: cuda
GPU: NVIDIA GeForce RTX 3060
CUDA (built with): 13.0
VRAM free/total: 11.8 / 12.9 GB
